# Motivational Plots

## 0. Imports & Set-Up:

In [ ]:
from pathlib import Path


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans


def show(fig):
    """Display a figure exactly once, then close it.

    A figure left open is rendered by the inline backend's end-of-cell
    flush, and a figure returned from a cell is rendered as that cell's
    result. A helper that does both renders twice in some cells and once in
    others. Displaying explicitly and closing is the one path that draws
    every figure exactly once.
    """
    display(fig)
    plt.close(fig)


In [ ]:
RANDOM_SEED = 42


In [7]:
from scipy.optimize import linear_sum_assignment


def align_cluster_labels(
    reference_labels: np.ndarray,
    labels: np.ndarray,
) -> np.ndarray:
    """
    Reorder labels to optimally match reference_labels using Hungarian assignment.
    """
    # Create common label space
    all_classes = np.union1d(reference_labels, labels)

    # Create square contingency matrix covering all class labels
    cont = np.zeros((len(all_classes), len(all_classes)), dtype=int)

    ref_to_idx = {val: i for i, val in enumerate(all_classes)}
    lab_to_idx = {val: i for i, val in enumerate(all_classes)}

    for r, l in zip(reference_labels, labels):
        cont[ref_to_idx[r], lab_to_idx[l]] += 1

    # Solve linear sum assignment
    row_ind, col_ind = linear_sum_assignment(-cont)

    # row_ind: indices in all_classes for reference
    # col_ind: indices in all_classes for predicted labels
    # Want to map: predicted class (at col_ind) -> reference class (at row_ind)
    mapping = {all_classes[c]: all_classes[r] for r, c in zip(row_ind, col_ind)}

    # Map original labels using 1-to-1 dictionary
    aligned = np.array([mapping[lbl] for lbl in labels], dtype=reference_labels.dtype)
    return aligned

In [9]:
%env PYTHONWARNINGS=ignore::FutureWarning,ignore::DeprecationWarning,ignore::UserWarning,ignore::RuntimeWarning

env: PYTHONWARNINGS=ignore::FutureWarning,ignore::DeprecationWarning,ignore::UserWarning,ignore::RuntimeWarning


---

## 1. Pre-Process the Data

### 1.1 Levine:

In [ ]:
from benchmarks.datasets import load_levine32

# Manuscript line 627: a stratified subsample of 5,000 cells.
X_levine, y_levine, meta_levine = load_levine32(subsample=5000)


### 1.2 Klein:

In [ ]:
from benchmarks.datasets import load_klein

X_klein, y_klein, meta_klein = load_klein(root=Path("../../data"))


---

## 2. Motivational Plots

In [53]:
levine_clusters_6 = KMeans(n_clusters=6, random_state=RANDOM_SEED).fit_predict(X_levine)
levine_clusters_10 = KMeans(n_clusters=10, random_state=RANDOM_SEED).fit_predict(
    X_levine
)
levine_clusters_14 = KMeans(n_clusters=14, random_state=RANDOM_SEED).fit_predict(
    X_levine
)
levine_clusters_18 = KMeans(n_clusters=18, random_state=RANDOM_SEED).fit_predict(
    X_levine
)

klein_clusters_2 = KMeans(n_clusters=2, random_state=RANDOM_SEED).fit_predict(X_klein)
klein_clusters_3 = KMeans(n_clusters=3, random_state=RANDOM_SEED).fit_predict(X_klein)
klein_clusters_4 = KMeans(n_clusters=4, random_state=RANDOM_SEED).fit_predict(X_klein)
klein_clusters_5 = KMeans(n_clusters=5, random_state=RANDOM_SEED).fit_predict(X_klein)

In [56]:
levine_clusters_6 = align_cluster_labels(levine_clusters_14, levine_clusters_6)
levine_clusters_10 = align_cluster_labels(levine_clusters_14, levine_clusters_10)
levine_clusters_18 = align_cluster_labels(levine_clusters_14, levine_clusters_18)

klein_clusters_2 = align_cluster_labels(klein_clusters_4, klein_clusters_2)
klein_clusters_3 = align_cluster_labels(klein_clusters_4, klein_clusters_3)
klein_clusters_5 = align_cluster_labels(klein_clusters_4, klein_clusters_5)

In [57]:
np.unique(klein_clusters_4)

array([0, 1, 2, 3], dtype=int32)

In [58]:
label_map = {0: 3, 1: 2, 2: 0, 3: 1, 4: 4}

klein_clusters_2 = np.array([label_map.get(lbl, lbl) for lbl in klein_clusters_2])
klein_clusters_3 = np.array([label_map.get(lbl, lbl) for lbl in klein_clusters_3])
klein_clusters_4 = np.array([label_map.get(lbl, lbl) for lbl in klein_clusters_4])
klein_clusters_5 = np.array([label_map.get(lbl, lbl) for lbl in klein_clusters_5])

In [59]:
np.unique(klein_clusters_3)

array([0, 1, 3])

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

from benchmarks.figures import DatasetRow, figure_clustering_problem

levine_embedding = TSNE(random_state=RANDOM_SEED).fit_transform(X_levine)
klein_embedding = PCA(n_components=2, random_state=RANDOM_SEED).fit_transform(X_klein)

# Marker areas are given for the 16-inch layout and scaled to the print width
# by the figure. It writes clustering_problem.png under vis/motivational_plots/
# and Fig1.tif under vis/plos/. Copying the PNG into the manuscript tree stays
# a manual step.
rows = [
    DatasetRow(
        name="Levine",
        embedding=levine_embedding,
        partitions={
            6: levine_clusters_6,
            10: levine_clusters_10,
            14: levine_clusters_14,
            18: levine_clusters_18,
        },
        marker_size=12,
    ),
    DatasetRow(
        name="Klein",
        embedding=klein_embedding,
        partitions={
            2: klein_clusters_2,
            3: klein_clusters_3,
            4: klein_clusters_4,
            5: klein_clusters_5,
        },
        marker_size=30,
    ),
]
show(figure_clustering_problem(rows))

---